In [ ]:
# -*- coding: utf-8 -*-
import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ---------- Configurações ----------
CAMINHO_XLSX = r'./Banco_de_Fardos.xlsx'   # <-- ajuste o path aqui (use r'....')
COLS_USAR = ['EAN','MATERIAL','NOME_CONC','CONVERSAO','FATOR']  # colunas esperadas
USE_ALL_FOR_TRAINING = False   # True -> usa 100% para treino (sem separar teste)
TEST_SIZE = 0.2                # usado quando USE_ALL_FOR_TRAINING = False
RANDOM_STATE = 42
# ------------------------------------

# função para extrair "pack counts" (C/12, CX12, x12, 12 UN, etc.)
def extract_pack_count(s: str):
    if pd.isna(s):
        return None
    text = str(s).upper()
    patterns = [
        r'C[^\d]*(\d{1,4})',        # C/12, C 12, C12
        r'CX[^\d]*(\d{1,4})',       # CX12
        r'(\d{1,4})\s*[X×\*]\s*\d*',# 12x or 12 x 30
        r'X\s*(\d{1,4})',           # x12
        r'C/\s*(\d{1,4})',          # C/12
        r'(\d{1,4})\s*UN\b',        # 12 UN
        r'(\d{1,4})\s*UNID\b',
        r'(\d{1,4})\s*COMP\b',
        r'(\d{1,4})\s*COMPRIMIDOS\b',
        r'(\d{1,4})\s*G\b',         # counts like '30 G' (cuidado)
    ]
    for p in patterns:
        m = re.search(p, text)
        if m:
            try:
                v = int(m.group(1))
                # filtro razoável: pequenos pacotes, não 200000 etc.
                if 1 < v <= 10000:
                    return v
            except:
                pass
    return None

# função para remover números de medidas (ML, G, KG, etc.) antes do TF-IDF
def remove_measurements(text: str):
    if pd.isna(text):
        return ''
    s = str(text).upper()
    # remove padrões tipo "2000ML", "400 ML", "250G", "C/12", "CX12", "12X30"
    s = re.sub(r'(\d{2,5}(\.\d+)?)(\s?)(ML|G|GR|GRS|KG|MG|L)\b', ' ', s)
    s = re.sub(r'\bC/?\s*\d{1,4}\b', ' ', s)
    s = re.sub(r'\bCX\s*\d{1,4}\b', ' ', s)
    s = re.sub(r'\b\d{1,4}\s*[X×\*]\s*\d{0,4}\b', ' ', s)
    s = re.sub(r'\b\d{1,4}\s*(UN|UNID|COMP|COMP\.)\b', ' ', s)
    # remove numbers isolados maiores (evitar tokens 2000)
    s = re.sub(r'\b\d{3,}\b', ' ', s)
    # normaliza espaços
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# ----------------- 1) Ler a planilha -----------------
tabela = pd.read_excel(CAMINHO_XLSX, dtype=str)  # lemos como str para não transformar
tabela.columns = tabela.columns.str.strip()      # limpa espaços nas colunas

# Verifica se colunas estão presentes
for c in COLS_USAR:
    if c not in tabela.columns:
        raise KeyError(f"Coluna esperada ausente no arquivo: {c}")

# Keep only the desired cols (safe)
tabela = tabela[COLS_USAR].copy()

# Convert FATOR to numeric when available (some rows NaN)
tabela['FATOR'] = pd.to_numeric(tabela['FATOR'], errors='coerce')

# ----------------- 2) Extrair padrões numéricos (packs) -----------------
tabela['pack_material'] = tabela['MATERIAL'].apply(extract_pack_count)
tabela['pack_nome'] = tabela['NOME_CONC'].apply(extract_pack_count)

# ----------------- 3) Criar texto limpo para TF-IDF (remove medidas) ----------
tabela['text_clean'] = (tabela['MATERIAL'].fillna('') + ' ' + tabela['NOME_CONC'].fillna('')).apply(remove_measurements)

# ----------------- 4) Features numéricas simples -----------------
# pack difference, presence flags
tabela['pack_material_f'] = tabela['pack_material'].fillna(0).astype(int)
tabela['pack_nome_f'] = tabela['pack_nome'].fillna(0).astype(int)
tabela['pack_diff'] = tabela['pack_material_f'] - tabela['pack_nome_f']
tabela['has_pack_material'] = (tabela['pack_material_f'] > 0).astype(int)
tabela['has_pack_nome'] = (tabela['pack_nome_f'] > 0).astype(int)

# ----------------- 5) Prepara X (TF-IDF + features) -----------------
tfv = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=3)
X_text = tfv.fit_transform(tabela['text_clean'].astype(str))

# features numéricas
num_feats = tabela[['pack_material_f','pack_nome_f','pack_diff','has_pack_material','has_pack_nome']].astype(float).values

# junta sparse + dense
X = hstack([X_text, num_feats])

# ----------------- 6) Targets -----------------
# Conversao -> label encoder
le_conv = LabelEncoder()
y_conv = le_conv.fit_transform(tabela['CONVERSAO'].fillna('DESCONHECIDO').astype(str))

# Fator -> se NaN preenche com 1.0 temporariamente (o regressor aprende)
y_fator = tabela['FATOR'].fillna(tabela['FATOR'].median()).astype(float).values

# ----------------- 7) Treino / Teste (opcional) -----------------
if USE_ALL_FOR_TRAINING:
    X_train = X
    y_conv_train = y_conv
    y_fator_train = y_fator
else:
    X_train, X_test, y_conv_train, y_conv_test, y_fator_train, y_fator_test = train_test_split(
        X, y_conv, y_fator, test_size=TEST_SIZE, random_state=RANDOM_STATE)

# ----------------- 8) Treinar modelos -----------------
clf_conv = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
reg_fator = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)

clf_conv.fit(X_train, y_conv_train)
reg_fator.fit(X_train, y_fator_train)

# Opcional: avaliar se você separou teste
if not USE_ALL_FOR_TRAINING:
    acc = clf_conv.score(X_test, y_conv_test)
    from sklearn.metrics import mean_absolute_error
    y_fpred = reg_fator.predict(X_test)
    mae = mean_absolute_error(y_fator_test, y_fpred)
    print(f"Validação — accuracy conversão: {acc:.3f}, MAE fator: {mae:.3f}")

# ----------------- 9) Prever para novos (aqui usamos mesma tabela como exemplo) ------------
# (no seu fluxo, substitua 'tabela' por 'novos_fardos' com as mesmas colunas e features)
X_novos = X
pred_conv = clf_conv.predict(X_novos)
pred_conv_labels = le_conv.inverse_transform(pred_conv)
pred_fator = reg_fator.predict(X_novos)

# ----------------- 10) Regras pos-processamento (filtro/override) -----------------
# Regras básicas:
# - se conversao == 'excluir' => fator = 0
# - se NOME_CONC tem pack_count e MATERIAL não tem -> aplicar regra (ex.: divide por pack_count)
# - se MATERIAL tem pack_count e NOME_CONC não -> aplicar multiplicador pack_count
# Observação: ajuste a lógica abaixo para o comportamento que você quiser.
resultado = tabela[['EAN','MATERIAL','NOME_CONC']].copy()
resultado['CONVERSAO'] = pred_conv_labels
resultado['FATOR'] = pred_fator

# normaliza nome da conversao (exemplos possíveis: 'multiplica','divide','excluir')
resultado['CONVERSAO'] = resultado['CONVERSAO'].str.lower()

for i, row in resultado.iterrows():
    conv = row['CONVERSAO']
    pm = tabela.at[i,'pack_material']
    pn = tabela.at[i,'pack_nome']
    # Se o modelo previu excluir -> forçar fator 0
    if conv in ['excluir','excluir ', 'exclui']:
        resultado.at[i,'FATOR'] = 0.0
        continue
    # Regra que você solicitou:
    # Se NOME_CONC tem pack_count e MATERIAL não tem -> divide por pack_count
    if pn and not pm:
        resultado.at[i,'CONVERSAO'] = 'divide'
        resultado.at[i,'FATOR'] = float(pn)
        continue
    # Se MATERIAL tem pack_count e NOME_CONC não -> multiplica por pack_count
    if pm and not pn:
        resultado.at[i,'CONVERSAO'] = 'multiplica'
        resultado.at[i,'FATOR'] = float(pm)
        continue
    # Se ambos tem packs, você pode decidir uma política:
    if pm and pn:
        # Exemplo simples: se pm == pn -> deixa fator 1.0 (mesma embalagem)
        if pm == pn:
            resultado.at[i,'CONVERSAO'] = 'multiplica'
            resultado.at[i,'FATOR'] = 1.0
        else:
            # Você pode escolher dividir ou multiplicar pela razão:
            # aqui definimos fator = pm/pn (ajuste conforme sua regra de negócio)
            try:
                resultado.at[i,'FATOR'] = float(pm) / float(pn)
                resultado.at[i,'CONVERSAO'] = 'multiplica' if (pm/pn) > 1 else 'divide'
            except:
                pass
        continue
    # limite de segurança: se fator previso for absurdo, ajusta
    f = resultado.at[i,'FATOR']
    if pd.notna(f):
        if f < 0:
            resultado.at[i,'FATOR'] = abs(f)
        if f > 100000:  # limite alto, ajuste conforme necessário
            resultado.at[i,'FATOR'] = 1.0

# ----------------- 11) Salvar resultado -----------------
SAIDA = r'./PlanilhaAtualizada_Resultados.xlsx'  # ajuste
resultado.to_excel(SAIDA, index=False)
print(f"Resultado salvo em: {SAIDA}")

# mostra as primeiras linhas
display(resultado.head(20))
